In [ ]:
import os
import yaml
from IPython.display import display, HTML
from typing import Optional

# --- Configuration ---
GENERATED_COURSE_DIR = 'generated_course'
MANIFEST_FILE = os.path.join(GENERATED_COURSE_DIR, '_course_manifest.yaml')
GENERATOR_NOTEBOOK_URL = 'generate_course.ipynb' # URL to redirect if no course exists

# --- Logic ---

def get_lo_to_open() -> Optional[str]:
    import jupyter_server.serverapp
    import requests

    # Get a list of all running servers
    servers = list(jupyter_server.serverapp.list_running_servers())

    if not servers:
        raise RuntimeError("No running Jupyter server found!")

    # Get info for the first server
    server_info = servers[0]
    host = server_info['hostname']
    port = server_info['port']
    base_url = server_info['base_url'] # e.g., '/' or '/lab'
    token = server_info['token']

    # Clean up base_url (it sometimes has a trailing /)
    if not base_url.endswith('/'):
        base_url += '/'

    # This is the base URL for any API calls
    api_base_url = f"http://{host}:{port}{base_url}"
    url = api_base_url + "q-toolkit/track_course"


    headers = {
        "Content-Type": "application/json",
        "Authorization": "token " + token
    }

    completed_los = []

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        completed_los = data.get('tracking_data', {}).get('completed_learning_objects', [])
    else:
        print(f"Failed to get course tracking info: {response.status_code} {response.text}")
        return None
    
    learning_objects = []
    with open(MANIFEST_FILE, 'r') as f:
        manifest = yaml.safe_load(f)
        learning_objects = manifest.get('course_chronology', [])

    incomplete_los = [lo for lo in learning_objects if lo not in completed_los]

    return incomplete_los[0] if incomplete_los else None


def redirect_browser(url: str):
    """
    Generates HTML with JavaScript to redirect the user's browser.
    """
    url = '/voila/render/' + url
    print('-------------------- ', url)
    js_redirect_code = f"window.location.href = '{url}';"
    display(HTML(f"""
        <p>Redirecting...</p>
        <script type='text/javascript'>
            {js_redirect_code}
        </script>
    """))

# --- Main Execution Logic ---

# Check if a course has been generated
if not os.path.exists(MANIFEST_FILE):
    # No manifest found, redirect to the generator page
    print(f"No course found at '{MANIFEST_FILE}'. Redirecting to generator...")
    redirect_browser(GENERATOR_NOTEBOOK_URL)
else:
    # Manifest exists, find which LO notebook to open
    start_lo_id = get_lo_to_open()
    
    if start_lo_id:
        # Construct the URL to the specific LO notebook
        # Assumes Voila serves relative to where it was launched
        lo_notebook_url = os.path.join(GENERATED_COURSE_DIR, start_lo_id, f"{start_lo_id}.ipynb")
        print(f"Course found. Redirecting to starting lesson: {start_lo_id}...")
        redirect_browser(lo_notebook_url)
    else:
        # Manifest existed but was invalid or empty
        display(HTML(f"<h3 style='color: red;'>Error reading course</h3>"
                     f"<p>Could not determine the starting lesson from <code>{MANIFEST_FILE}</code>.</p>"
                     f"<p>You may need to regenerate the course using <a href='{GENERATOR_NOTEBOOK_URL}'>the generator</a>.</p>"))


Course found. Redirecting to starting lesson: LO-3.1...
--------------------  /voila/render/generated_course/LO-3.1/LO-3.1.ipynb
